In [1]:
import geopandas as gpd
import gcsfs
import google.auth
import numpy as np
import pandas as pd

import world_cup_vars as wc_vars

import altair as alt
import folium
import polars as pl
from great_tables import GT

credentials, _ = google.auth.default()

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

In [2]:
import C1_service_by_route as C1
import C4_event_helpers as C4

In [3]:
stadium_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}points_of_interest_{wc_vars.event_name}.parquet", 
    storage_options = {"token": credentials}
)

In [8]:
socal_routes = np.concatenate(
    [i for i in wc_vars.special_socal_routes_dict.values() if i is not None]
).ravel()

In [7]:
socal_route_gdf = pd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_schedule_rt_route_direction_summary_world_cup.parquet", 
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["service_date", "schedule_name", "feed_key", "route_id", "route_id_cleaned",
               "route_name", "direction_id", "route_type",
               "shape_id", "shape_array_key", 
               "n_trips", "num_stop_times"
              ],
    filters = [[
        ("schedule_name", "in", wc_vars.socal_names),
        ("route_name", "in", socal_routes)
    ]]
).pipe(
    C1.merge_routes_with_shape_geom
).pipe(
    C4.tag_event_days_and_times, 
    wc_vars.sofi_match_times
)

In [11]:
socal_route_ids = socal_route_gdf.route_id.unique()

In [30]:
metric_cols = [
    #"n_hours_in_service",
    "arrivals_per_hour_owl", "arrivals_per_hour_early_am", 
    "arrivals_per_hour_am_peak", "arrivals_per_hour_midday",
    "arrivals_per_hour_pm_peak", "arrivals_per_hour_evening",
    "arrivals_owl", "arrivals_early_am",
    "arrivals_am_peak", "arrivals_midday",
    "arrivals_pm_peak", "arrivals_evening", 
    "route_id_array", #"route_type_array",
    #"wheelchair_boarding", "location_type"
]

In [31]:
def get_stops_along_special_routes(
    stop_gdf: gpd.GeoDataFrame,
    list_of_routes: list
) -> pd.DataFrame:
    # filter stops to ones that travel along the routes we want
    # explode to see which route_ids, then drop the ones that aren't found in our list of service changes
    keep_cols = ["feed_key", "stop_id", "stop_name"]
    
    stops_for_special_routes = (
        stop_gdf[keep_cols + ["route_id_array"]]
        .explode("route_id_array")
        .query('route_id_array in @list_of_routes')
        [keep_cols]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    
    return stops_for_special_routes

In [32]:
stops_near_sofi = pd.read_parquet(
    f"{GCS_FILE_PATH}stops_near_poi.parquet", 
    filesystem = gcsfs.GCSFileSystem(),
    filters = [[("schedule_name", "in", wc_vars.socal_names)]]
)

stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_scheduled_stops_{wc_vars.event_name}.parquet",
    storage_options = {"token": credentials},
    columns = ["service_date", "feed_key", "stop_id", "stop_name",
               "daily_arrivals",
              "geometry"] + metric_cols,
).merge(
    stops_near_sofi,
    on = ["feed_key", "stop_id", "stop_name"],
    how = "inner"
)

In [33]:
stops_for_special_routes = get_stops_along_special_routes(stop_gdf, socal_route_ids)

In [40]:
stop_gdf2 = pd.merge(
    stop_gdf,
    stops_for_special_routes,
    on = ["feed_key", "stop_id", "stop_name"],
    how = "inner"
).pipe(C4.tag_event_days_and_times, wc_vars.sofi_match_times)

In [42]:
time_of_day_buckets = ["owl", "early_am", "am_peak", "midday", "pm_peak", "evening"]

arrivals_by_event_type = (
    stop_gdf2
    .groupby(["schedule_name", #"feed_key",
              "stop_id", "stop_name", 
              "event_day", "day_type"])
    .agg({
        "daily_arrivals": "sum",
        "service_date": "nunique",
        #**{f"arrivals_per_hour_{t}": "sum" 
        #for t in time_of_day_buckets},
    }).reset_index()
    # rename columns here for clarity
    .rename(columns = {
        "daily_arrivals": "total_arrivals",
        "service_date": "n_days",
    })
)

arrivals_by_event_type = arrivals_by_event_type.assign(
    daily_arrivals = arrivals_by_event_type.total_arrivals.divide(
        arrivals_by_event_type.n_days).round(2),
)#.merge(
 #   routes_to_explode,
 #   on = ["schedule_name", "stop_id", "stop_name"],
 #   how = "inner"
#)

In [45]:
arrivals_by_event_type[arrivals_by_event_type.stop_name == "LAX / Metro Transit Center"]

,schedule_name,stop_id,stop_name,event_day,day_type,total_arrivals,n_days,daily_arrivals
315,LA Metro Rail Schedule,80702,LAX / Metro Transit Center,False,weekday,8931,22,405.95
316,LA Metro Rail Schedule,80702,LAX / Metro Transit Center,False,weekend,3420,9,380.00
317,LA Metro Rail Schedule,80702,LAX / Metro Transit Center,True,weekday,2994,6,499.00
318,LA Metro Rail Schedule,80702,LAX / Metro Transit Center,True,weekend,886,2,443.00


In [86]:
# show change (switch to wide)

pivoted = arrivals_by_event_type[arrivals_by_event_type.stop_name == "LAX / Metro Transit Center"].pivot(
    index=['schedule_name', 'stop_id', 'stop_name'], 
    columns=['event_day', 'day_type'], 
    values='daily_arrivals'
).reset_index()

# create a list of the new column names in the right order
new_cols=[f'{tup[1]}_{tup[0]}' for tup in pivoted.columns]

# assign it to the dataframe (assuming you named it pivoted
pivoted.columns = new_cols

In [93]:
pivoted = arrivals_by_event_type[arrivals_by_event_type.stop_name == "LAX / Metro Transit Center"].pivot(
    index=['schedule_name', 'stop_id', 'stop_name'], 
    columns=['event_day', 'day_type'], 
    values='daily_arrivals'
).reset_index()

In [ ]:
pivoted.columns.get_level_values(0)


In [97]:
pivoted.columns.get_level_values(0)
pivoted.columns.get_level_values(1)

Index(['', '', '', 'weekday', 'weekend', 'weekday', 'weekend'], dtype='object', name='day_type')

In [88]:
from gtfs_curator_utils import sql

In [70]:
pivoted = arrivals_by_event_type[arrivals_by_event_type.stop_name == "LAX / Metro Transit Center"].pivot(
    index=['schedule_name', 'stop_id', 'stop_name'], 
    columns=['event_day', 'day_type'], 
    values='daily_arrivals'
).reset_index()

In [85]:
pivoted.columns

MultiIndex([('schedule_name',        ''),
            (      'stop_id',        ''),
            (    'stop_name',        ''),
            (          False, 'weekday'),
            (          False, 'weekend'),
            (           True, 'weekday'),
            (           True, 'weekend')],
           names=['event_day', 'day_type'])

In [47]:
arrivals_by_event_type.stop_name.unique()

array(['LAX/METRO TRANSIT CENTER', 'ARBOR VITAE ST & BELLANCA AVE',
       'WESTCHESTER PKWY & AIRPORT BLVD', 'LAX/MTC BAY 16 DROP OFF ONLY',
       'LOS ANGELES STADIUM - MANCHESTER & PRAIRIE - LOT T',
       'Arbor Vitae/Bellanca', 'Westchester/Airport',
       'LAX/Metro Transit Center Bay 7',
       'LAX Metro Transit Center Bay 16',
       'LAX/Metro Transit Center Bay 16', 'Western Ave & 120th St',
       'Imperial Hwy & Denker Ave', 'Imperial Hwy & Normandie Ave',
       'Imperial Hwy & Budlong Ave', 'Imperial Hwy & Western Ave',
       'Western Ave & College Dr', 'El Segundo Bl & Shoup Ave',
       'El Segundo Bl & Inglewood Ave', 'El Segundo Bl & Ramona Ave',
       'El Segundo Bl & Hawthorne Bl', 'El Segundo Bl & Birch Ave',
       'El Segundo Bl & Oxford Ave', 'El Segundo Bl & Prarie Ave',
       'El Segundo Bl & Cordary Ave', 'El Segundo Bl & Yukon Ave',
       'El Segundo Bl & Cerise Ave', 'El Segundo Bl & Chadron Ave',
       'El Segundo Bl & Crenshaw Bl', 'El Segundo Bl 

In [46]:
alt.Chart(arrivals_by_event_type[arrivals_by_event_type.stop_name == "LAX / Metro Transit Center"]).mark_bar().encode(
    x="daily_arrivals:Q",
    y="event_day:N",
    color=alt.Color("event_day:N", scale=alt.Scale(scheme="viridis")),
    column="day_type:N",
).properties(height=75, width=300)

alt.Chart(...)